# IE2026 Task 1b — QLoRA 3k Training, FIXED (Qwen2.5-VL-7B)

**Requires T4×2 GPU. Single session: ~7h train + ~2h checkpoint selection.**

**Fixes vs previous 3k run (which scored devtest CI 0.036, worse than the 2k run's 0.032):**

1. **Full data coverage** — previous run resumed across 3 sessions, each with a fresh
   *unseeded* shuffle and no skipping of already-consumed samples. Expected unique items
   seen ≈ ~2,030 of 3,000 (many duplicated, ~970 never seen) — the 3k advantage never
   existed. Now: epoch order is a seeded permutation, and resume skips exactly the
   samples already consumed, so all 3,000 items are seen exactly once per epoch.
2. **No patience-1 early stop** — previous run stopped at step 700 because dev CI got
   worse by exactly ONE item (0.022→0.024; dev granularity is 0.002). That is noise.
   Now: train the full 750-step cosine schedule to the end (like the 2k run did).
3. **Checkpoint selection at deployment config** — previous run picked the checkpoint by
   dev CI measured at 256px with the training prompt, but submission inference runs at
   1024px with a single-user-message prompt. Now: after training, candidate checkpoints
   are scored on dev at **1024px with the exact inference prompt/parse**, ties within
   0.004 go to the *later* (more LR-annealed) checkpoint.
4. **Global seeding** (`random`/`numpy`/`torch`) for reproducibility.
5. **No hard-coded HF token** — reads `HF_TOKEN` from Kaggle secrets.
   (The old notebook leaked a write token in plain text — **revoke it**.)


## 1. Install

In [ ]:
import os, warnings, time
warnings.filterwarnings('ignore')
for _major in ('12', '13'):
    _src = f'/usr/local/cuda/lib64/libnvJitLink.so.{_major}'
    _dst = '/usr/local/cuda/lib64/libnvJitLink.so.13'
    if os.path.exists(_src) and not os.path.exists(_dst):
        os.symlink(_src, _dst); print(f'Symlinked .{_major} -> .13'); break
os.environ['BITSANDBYTES_NOWELCOME'] = '1'
os.environ['BNB_CUDA_VERSION']        = '128'
os.environ['TOKENIZERS_PARALLELISM']  = 'false'
!pip install -q -U 'transformers>=4.49.0' 'peft>=0.10.0' \
    accelerate bitsandbytes qwen-vl-utils 2>&1 | tail -4
print('Dependencies ready.')

## 2. Configuration

In [ ]:
import os, random, torch
import numpy as np
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# ── Dataset ──────────────────────────────────────────────────────────────
REPO_ID  = 'QCRI/AynVQA-ArabicNLP26'
TASK     = 'task1b'
LANG     = 'en'
SPLIT    = 'train'

# ── Model ────────────────────────────────────────────────────────────────
VLM_MODEL     = 'Qwen/Qwen2.5-VL-7B-Instruct'
MAX_PIXELS    = 256 * 28 * 28    # training resolution (~244 image tokens)

# ── LoRA (LLM layers only) ────────────────────────────────────────────────
LORA_RANK     = 8
LORA_ALPHA    = 16
LORA_DROPOUT  = 0.05
LORA_TARGETS  = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                 'gate_proj', 'up_proj', 'down_proj']

# ── Training ─────────────────────────────────────────────────────────────
SEED          = 42
MAX_STEPS     = 750    # 3000 items / 4 GRAD_ACCUM = 750 opt steps = exactly 1 epoch
GRAD_ACCUM    = 4
BATCH_SIZE    = 1
LEARNING_RATE = 2e-4
MAX_SEQ_LEN   = 1280
WARMUP_RATIO  = 0.05
SAVE_STEPS    = 150    # checkpoints at 150/300/450/600/750
LOGGING_STEPS = 10
NUM_EPOCHS    = 1

# FIX 2: no early stopping. Optional informational dev-CI log at each
# checkpoint (256px, training prompt). Costs ~12 min per checkpoint —
# leave False to keep the session comfortably inside Kaggle's 12h limit.
LOG_CI_AT_CKPT = False

# ── Checkpoint selection (runs AFTER training, FIX 3) ─────────────────────
# Candidates scored on dev at deployment config (1024px, inference prompt).
CANDIDATE_STEPS  = [450, 600, 750]
EVAL_MAX_PIXELS  = 1024 * 28 * 28   # must match the inference notebook
TIE_EPS          = 0.004            # dev CI diffs <= 2 items are a tie -> prefer later step

# ── Resume (only if the session died mid-training) ────────────────────────
# Upload /kaggle/working/checkpoints/step_N as a dataset, then point here.
# Data order is seeded, so the resumed session skips exactly the samples
# already consumed (FIX 1) — no re-shuffling, no double-seen items.
# NOTE: checkpoints from the OLD (unfixed) notebooks are not compatible
# with this bookkeeping — start fresh.
RESUME_FROM   = None   # e.g. '/kaggle/input/qlora-ckpt/step_450'
RESUME_STEP   = 0      # must match the checkpoint step number

# ── Output ───────────────────────────────────────────────────────────────
OUTPUT_DIR    = '/kaggle/working/checkpoints'
FINAL_ADAPTER = '/kaggle/working/adapter_final'

# FIX 4: seed everything
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

N_GPUS = torch.cuda.device_count()
assert N_GPUS >= 1, 'No GPU found'
print(f'GPUs: {N_GPUS}')
for i in range(N_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name} ({p.total_memory/1024**3:.1f} GB)')
print(f'MAX_PIXELS:   {MAX_PIXELS} (~{MAX_PIXELS//784} image tokens)')
print(f'MAX_STEPS:    {MAX_STEPS}')
print(f'SAVE_STEPS:   {SAVE_STEPS}')
print(f'Candidates:   {CANDIDATE_STEPS} (scored at {EVAL_MAX_PIXELS//784} img tokens)')
print(f'Resume from:  {RESUME_FROM or "scratch"}')
print(f'Est. at 33s/step: {(MAX_STEPS-RESUME_STEP)*33/3600:.1f}h train '
      f'+ ~{len(CANDIDATE_STEPS)*0.7:.1f}h selection')

In [ ]:
# FIX 5: token from Kaggle secrets (Add-ons -> Secrets -> HF_TOKEN).
# Falls back to anonymous access if the secret is missing.
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret('HF_TOKEN'))
    print('Logged in to HF via Kaggle secret.')
except Exception as e:
    print(f'No HF_TOKEN secret ({e}) — continuing anonymously.')

## 3. Load training data + download images

In [ ]:
import json
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

jsonl = hf_hub_download(
    REPO_ID, filename=f'{TASK}/{SPLIT}_{LANG}.jsonl', repo_type='dataset')
all_records = [json.loads(l) for l in open(jsonl, encoding='utf-8') if l.strip()]
print(f'Total records: {len(all_records)} | has labels: {"labels" in all_records[0]}')

records = all_records  # use everything, no subsampling
print(f'Using all {len(records)} records for {NUM_EPOCHS} epochs '
      f'-> {len(records) * NUM_EPOCHS // GRAD_ACCUM} opt steps (target {MAX_STEPS})')

needed = sorted({r['image'] for r in records})
img_paths = {}
failed = []
for rel in tqdm(needed, desc='images'):
    try:
        img_paths[rel] = hf_hub_download(
            REPO_ID, filename=rel, repo_type='dataset')
    except Exception as e:
        failed.append(rel)
        print(f'Failed: {rel}: {e}')
print(f'Downloaded {len(img_paths)}/{len(needed)} images.')

## 4. Dataset + deterministic DataLoader (FIX 1)

Epoch order = `randperm` seeded with `SEED + epoch`, identical in every session.
On resume, the first `consumed` samples of the current epoch's order are skipped, so
across sessions every item is seen **exactly once per epoch** — the old notebooks
re-shuffled from scratch on each resume and left ~1/3 of the 3k set unseen.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from transformers import AutoProcessor
from qwen_vl_utils import process_vision_info

SYSTEM_PROMPT = (
    'You are a visual fact-checker examining an image from the Arab world.\n'
    'Below are THREE statements about this image. '
    'Exactly ONE statement is grounded in the image (True). '
    'The other two are plausible-sounding hallucinations (False).'
)
USER_TEMPLATE = (
    'Statement 1: {s0}\n'
    'Statement 2: {s1}\n'
    'Statement 3: {s2}\n\n'
    'Instructions:\n'
    '- On the VERY FIRST line write ONLY: "Answer: X" where X is 1, 2, or 3.\n'
    '- For each statement evaluate:\n'
    '    (a) Colour/texture evidence for or against\n'
    '    (b) Shape/form evidence for or against\n'
    '    (c) Contextual evidence for or against\n'
    '- Then state your conclusion.\n'
    'Do not write anything before the Answer line.'
)

processor = AutoProcessor.from_pretrained(VLM_MODEL, max_pixels=MAX_PIXELS)

class HalDetectDataset(Dataset):
    def __init__(self, records, img_paths, processor, max_seq_len):
        self.samples = [
            {'image_path': img_paths[r['image']],
             'statements': r['statements'],
             'true_idx':   r['labels'].index(True)}
            for r in records if r['image'] in img_paths
        ]
        self.processor   = processor
        self.max_seq_len = max_seq_len
        print(f'Dataset: {len(self.samples)} samples '
              f'({len(records)-len(self.samples)} skipped)')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s   = self.samples[idx]
        target   = f'Answer: {s["true_idx"] + 1}'
        user_txt = USER_TEMPLATE.format(
            s0=s['statements'][0], s1=s['statements'][1], s2=s['statements'][2])
        messages = [
            {'role': 'system',    'content': SYSTEM_PROMPT},
            {'role': 'user',      'content': [
                {'type': 'image', 'image': s['image_path']},
                {'type': 'text',  'text':  user_txt}]},
            {'role': 'assistant', 'content': target},
        ]
        prompt_text = self.processor.apply_chat_template(
            messages[:-1], tokenize=False, add_generation_prompt=True)
        full_text   = self.processor.apply_chat_template(
            messages,       tokenize=False, add_generation_prompt=False)
        image_inputs, _ = process_vision_info(messages)

        enc_full   = self.processor(text=[full_text],   images=image_inputs,
                                    truncation=False, return_tensors='pt')
        enc_prompt = self.processor(text=[prompt_text], images=image_inputs,
                                    truncation=False, return_tensors='pt')

        input_ids  = enc_full['input_ids'][0][:self.max_seq_len]
        attn_mask  = enc_full['attention_mask'][0][:self.max_seq_len]
        prompt_len = enc_prompt['input_ids'].shape[1]

        pad_id  = self.processor.tokenizer.pad_token_id or 0
        pad_len = self.max_seq_len - input_ids.shape[0]
        if pad_len > 0:
            input_ids = torch.cat(
                [input_ids, torch.full((pad_len,), pad_id, dtype=torch.long)])
            attn_mask = torch.cat(
                [attn_mask, torch.zeros(pad_len, dtype=torch.long)])

        labels = input_ids.clone()
        labels[:prompt_len] = -100
        labels[input_ids == pad_id] = -100

        return {
            'input_ids':      input_ids,
            'attention_mask': attn_mask,
            'pixel_values':   enc_full['pixel_values'],
            'image_grid_thw': enc_full['image_grid_thw'],
            'labels':         labels,
        }

dataset = HalDetectDataset(records, img_paths, processor, MAX_SEQ_LEN)

ex = dataset[0]
print(f'input_ids:     {ex["input_ids"].shape}')
print(f'pixel_values:  {ex["pixel_values"].shape}')
n_tgt = (ex['labels'] != -100).sum().item()
print(f'target tokens: {n_tgt}')
print(f'decoded:       '
      f'{processor.decode(ex["labels"][ex["labels"] != -100], skip_special_tokens=True)}')

def collate_fn(batch):
    return {
        'input_ids':      torch.stack([b['input_ids']      for b in batch]),
        'attention_mask': torch.stack([b['attention_mask'] for b in batch]),
        'labels':         torch.stack([b['labels']         for b in batch]),
        'pixel_values':   torch.cat(  [b['pixel_values']   for b in batch], dim=0),
        'image_grid_thw': torch.cat(  [b['image_grid_thw'] for b in batch], dim=0),
    }

def make_epoch_loader(epoch_idx, skip_samples=0):
    """Deterministic per-epoch order; resume skips already-consumed samples."""
    g = torch.Generator().manual_seed(SEED + epoch_idx)
    order = torch.randperm(len(dataset), generator=g).tolist()
    if skip_samples:
        print(f'Epoch {epoch_idx}: skipping first {skip_samples} '
              f'already-consumed samples of the seeded order')
        order = order[skip_samples:]
    return DataLoader(
        Subset(dataset, order), batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=collate_fn, num_workers=2,
        prefetch_factor=2, persistent_workers=False, pin_memory=False)

steps_per_epoch = len(dataset) // GRAD_ACCUM
warmup_steps    = int(MAX_STEPS * WARMUP_RATIO)
print(f'\nSteps/epoch:      {steps_per_epoch}')
print(f'Target opt steps: {MAX_STEPS}')
print(f'Warmup steps:     {warmup_steps}')
print(f'Est. at 33s/step: {MAX_STEPS*33/3600:.1f}h total')

## 5. Load model + QLoRA + freeze vision encoder

**GPU fix:** after model load with `device_map='auto'`, `pixel_values` may land on
GPU 0 while LLM layers are on GPU 1 — track the LLM's primary device and move
`pixel_values` there before every forward pass.

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, \
    prepare_model_for_kbit_training, PeftModel

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=dtype, bnb_4bit_use_double_quant=True)

max_mem = {i: '13000MiB' for i in range(N_GPUS)}
max_mem['cpu'] = '4GiB'

print('Loading base model...')
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    VLM_MODEL, torch_dtype=dtype,
    device_map='auto', max_memory=max_mem,
    quantization_config=bnb_config)

if hasattr(base_model, 'hf_device_map'):
    from collections import Counter
    dist = dict(Counter(base_model.hf_device_map.values()))
    print(f'Layer distribution: {dist}')

LLM_DEVICE = next(base_model.lm_head.parameters()).device
print(f'LLM primary device: {LLM_DEVICE}  (pixel_values routed here)')

base_model = prepare_model_for_kbit_training(
    base_model, use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False})

lora_cfg = LoraConfig(
    r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS, bias='none', task_type=TaskType.CAUSAL_LM)

if RESUME_FROM:
    print(f'Resuming: loading adapter from {RESUME_FROM}')
    model = PeftModel.from_pretrained(base_model, RESUME_FROM, is_trainable=True)
    print('Adapter loaded — continuing training.')
else:
    model = get_peft_model(base_model, lora_cfg)

n_frozen = n_train = 0
for name, param in model.named_parameters():
    if 'visual' in name:
        param.requires_grad = False
        n_frozen += param.numel()
    elif param.requires_grad:
        n_train += param.numel()
print(f'Frozen  (vision encoder): {n_frozen:,}')
print(f'Trainable (LoRA on LLM):  {n_train:,}')
model.print_trainable_parameters()

for i in range(N_GPUS):
    a = torch.cuda.memory_allocated(i)/1024**3
    t = torch.cuda.get_device_properties(i).total_memory/1024**3
    print(f'GPU {i}: {a:.1f}/{t:.1f} GB')

## 5b. Dev set + CI scorer

`evaluate_ci()` reproduces the official Ayn-VQA `score_1b()` metrics. It now takes an
explicit processor and prompt mode so the **checkpoint-selection pass can score at the
exact deployment config** (1024px, single user message, regex parse — identical to the
inference notebook), instead of the 256px training config the old run selected on.

In [ ]:
import re, shutil
from qwen_vl_utils import process_vision_info

# ── Load dev split (labeled, 500 items) ───────────────────────────────────
dev_jsonl = hf_hub_download(
    REPO_ID, filename=f'{TASK}/dev_{LANG}.jsonl', repo_type='dataset')
dev_records = [json.loads(l) for l in open(dev_jsonl, encoding='utf-8') if l.strip()]
print(f'Dev records: {len(dev_records)}')

dev_needed = sorted({r['image'] for r in dev_records})
dev_img_paths = {}
for rel in tqdm(dev_needed, desc='dev images'):
    try:
        dev_img_paths[rel] = hf_hub_download(REPO_ID, filename=rel, repo_type='dataset')
    except Exception as e:
        print(f'Failed: {rel}: {e}')
print(f'Downloaded {len(dev_img_paths)}/{len(dev_needed)} dev images.')

dev_samples = [
    {'image_path': dev_img_paths[r['image']],
     'statements': r['statements'],
     'true_idx':   r['labels'].index(True)}
    for r in dev_records if r['image'] in dev_img_paths
]
print(f'Dev samples ready: {len(dev_samples)}')

# Exact prompt used by inference-qlora-q7b-frn.ipynb (single user message,
# system text folded in — NOT the system+user split used during training).
INFER_PROMPT = (
    'You are a visual fact-checker examining an image from the Arab world.\n'
    'Below are THREE statements about this image. '
    'Exactly ONE statement is grounded in the image (True). '
    'The other two are plausible-sounding hallucinations (False).\n\n'
    'Statement 1: {s0}\n'
    'Statement 2: {s1}\n'
    'Statement 3: {s2}\n\n'
    'Instructions:\n'
    '- On the VERY FIRST line write ONLY: "Answer: X" where X is 1, 2, or 3.\n'
    '- For each statement evaluate:\n'
    '    (a) Colour/texture evidence for or against\n'
    '    (b) Shape/form evidence for or against\n'
    '    (c) Contextual evidence for or against\n'
    '- Then state your conclusion.\n'
    'Do not write anything before the Answer line.'
)

def parse_answer(raw):
    """Same parser as the inference notebook."""
    lines = [l.strip() for l in raw.splitlines() if l.strip()]
    for line in lines:
        m = re.search(r'answer\s*[:\-]?\s*([123])', line, re.IGNORECASE)
        if m: return int(m.group(1))
        break
    for line in lines:
        m = re.search(r'answer\s*[:\-]?\s*([123])', line, re.IGNORECASE)
        if m: return int(m.group(1))
    return None

def _rate(n, d):
    return float(round(n / d, 6)) if d else 0.0

@torch.no_grad()
def evaluate_ci(model, proc, samples, inference_style=False, max_new_tokens=16,
                desc='dev eval'):
    """Official score_1b() metrics on `samples`.
    inference_style=False -> training prompt (system+user), first-digit parse.
    inference_style=True  -> deployment config: single user message with
    INFER_PROMPT + the inference notebook's regex parser."""
    model.eval()

    total = q_minus_total = 0
    q_plus_c = q_minus_c = combined_c = 0
    cfhr_2 = cfhr_2_total = cfhr_3 = cfhr_3_total = 0

    for s in tqdm(samples, desc=desc, leave=False):
        true_idx = s['true_idx']
        if inference_style:
            text = INFER_PROMPT.format(
                s0=s['statements'][0], s1=s['statements'][1], s2=s['statements'][2])
            messages = [{'role': 'user', 'content': [
                {'type': 'image', 'image': s['image_path']},
                {'type': 'text',  'text':  text}]}]
        else:
            user_txt = USER_TEMPLATE.format(
                s0=s['statements'][0], s1=s['statements'][1], s2=s['statements'][2])
            messages = [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': [
                    {'type': 'image', 'image': s['image_path']},
                    {'type': 'text',  'text':  user_txt}]},
            ]
        prompt_text = proc.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
        image_inputs, _ = process_vision_info(messages)
        enc = proc(text=[prompt_text], images=image_inputs,
                   truncation=False, return_tensors='pt')

        pv  = enc.pop('pixel_values').to(dtype).to(LLM_DEVICE)
        thw = enc.pop('image_grid_thw').to(LLM_DEVICE)
        enc = {k: v.to(LLM_DEVICE) for k, v in enc.items()}
        enc['pixel_values']   = pv
        enc['image_grid_thw'] = thw

        gen = model.generate(
            **enc, max_new_tokens=max_new_tokens, do_sample=False,
            pad_token_id=proc.tokenizer.pad_token_id)
        new_tokens = gen[0][enc['input_ids'].shape[1]:]
        decoded = proc.decode(new_tokens, skip_special_tokens=True)

        ans = parse_answer(decoded)
        pred_idx = ans - 1 if ans is not None else None

        # Reconstruct the 3 True/False judgments implied by pred_idx.
        # Missing/unparseable prediction counts as fully wrong on all three
        # judgments — matching the official scorer's 'missing -> wrong' rule.
        false_idx = [i for i in range(3) if i != true_idx]
        if pred_idx is None:
            inc_qp = inc_qm0 = inc_qm1 = False
        else:
            inc_qp  = (pred_idx == true_idx)
            inc_qm0 = (pred_idx != false_idx[0])
            inc_qm1 = (pred_idx != false_idx[1])

        total += 1
        q_minus_total += 2
        q_plus_c  += inc_qp
        q_minus_c += inc_qm0 + inc_qm1
        combined_c += inc_qp and inc_qm0 and inc_qm1

        if inc_qp:
            cfhr_2_total += 1
            if not (inc_qm0 and inc_qm1):
                cfhr_2 += 1
        if inc_qp or inc_qm0 or inc_qm1:
            cfhr_3_total += 1
            if not (inc_qp and inc_qm0 and inc_qm1):
                cfhr_3 += 1

    model.train()
    for module in model.modules():
        if 'Visual' in type(module).__name__:
            module.eval()

    return {
        'contrastive_instability': _rate(cfhr_3, cfhr_3_total),  # ranking metric, lower better
        'combined_accuracy':       _rate(combined_c, total),
        'cfhr':                    _rate(cfhr_2, cfhr_2_total),
        'q_plus_accuracy':         _rate(q_plus_c, total),
        'q_minus_accuracy':        _rate(q_minus_c, q_minus_total),
    }

## 6. Training — full 750-step schedule, no early stop (FIX 2)

Every `SAVE_STEPS` a checkpoint (adapter + optimizer + scheduler) is written.
Dev-CI logging at checkpoints is informational only (`LOG_CI_AT_CKPT`) — it never
stops training. Checkpoint choice happens in Section 7 at deployment config.

In [ ]:
import json
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

os.makedirs(OUTPUT_DIR,    exist_ok=True)
os.makedirs(FINAL_ADAPTER, exist_ok=True)

optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE, weight_decay=0.01)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=MAX_STEPS)

# Restore optimizer + scheduler state directly on resume (no warmup replay,
# no Adam-moment reset).
if RESUME_FROM and RESUME_STEP > 0:
    opt_path   = os.path.join(RESUME_FROM, 'optimizer.pt')
    sched_path = os.path.join(RESUME_FROM, 'scheduler.json')

    if os.path.exists(opt_path):
        # Load to CPU first, then move each state tensor to its own
        # parameter's device (device_map='auto' splits params across GPUs).
        saved_state = torch.load(opt_path, map_location='cpu')
        optimizer.load_state_dict(saved_state)
        for param, state in optimizer.state.items():
            for k, v in state.items():
                if torch.is_tensor(v) and v.device != param.device:
                    state[k] = v.to(param.device)
        print(f'Restored optimizer state from {opt_path}')
    else:
        print(f'WARNING: {opt_path} not found — Adam moments start from zero, '
              f'updates biased for the first ~50-100 steps.')

    if os.path.exists(sched_path):
        with open(sched_path) as f:
            scheduler.load_state_dict(json.load(f))
        print(f'Restored scheduler state from {sched_path} '
              f'(LR = {scheduler.get_last_lr()[0]:.2e})')
    else:
        print(f'WARNING: {sched_path} not found — LR schedule restarts.')

model.train()
for module in model.modules():
    if 'Visual' in type(module).__name__:
        module.eval()   # keep vision encoder in eval mode

global_step = RESUME_STEP
t0          = time.time()
step_times  = []
done        = global_step >= MAX_STEPS

print(f'Training: {MAX_STEPS} opt steps total | starting from step {global_step}')
print(f'Checkpointing every {SAVE_STEPS} steps\n')

if done:
    print('Already at MAX_STEPS — skip to Section 7 (checkpoint selection).')

while not done:
    # FIX 1: deterministic epoch order; resume skips consumed samples.
    epoch    = global_step // steps_per_epoch + 1
    consumed = (global_step % steps_per_epoch) * GRAD_ACCUM
    loader   = make_epoch_loader(epoch, skip_samples=consumed)

    epoch_loss = 0.0
    n_batches  = 0
    optimizer.zero_grad()

    pbar = tqdm(loader, desc=f'Epoch {epoch}')
    for step, batch in enumerate(pbar):
        t_step = time.time()

        # Route pixel_values to the LLM device before forward.
        pv  = batch.pop('pixel_values').to(dtype).to(LLM_DEVICE)
        thw = batch.pop('image_grid_thw').to(LLM_DEVICE)
        batch = {k: v.to(LLM_DEVICE) for k, v in batch.items()}
        batch['pixel_values']   = pv
        batch['image_grid_thw'] = thw

        outputs = model(**batch)
        loss    = outputs.loss / GRAD_ACCUM
        loss.backward()

        epoch_loss += loss.item() * GRAD_ACCUM
        n_batches  += 1
        step_times.append(time.time() - t_step)

        pbar.set_postfix(
            loss=f'{loss.item()*GRAD_ACCUM:.4f}',
            sps=f'{step_times[-1]:.1f}s',
            step=global_step)

        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

            if global_step % LOGGING_STEPS == 0:
                avg  = epoch_loss / n_batches
                lr_n = scheduler.get_last_lr()[0]
                sps  = sum(step_times[-40:]) / len(step_times[-40:])
                eta  = (MAX_STEPS - global_step) * sps * GRAD_ACCUM / 3600
                print(f'Step {global_step:4d}/{MAX_STEPS} | '
                      f'loss={avg:.4f} | lr={lr_n:.2e} | '
                      f'{sps:.1f}s/batch | ETA {eta:.1f}h')

            if global_step % SAVE_STEPS == 0:
                ckpt = os.path.join(OUTPUT_DIR, f'step_{global_step}')
                model.save_pretrained(ckpt)
                processor.save_pretrained(ckpt)
                torch.save(optimizer.state_dict(), os.path.join(ckpt, 'optimizer.pt'))
                with open(os.path.join(ckpt, 'scheduler.json'), 'w') as f:
                    json.dump(scheduler.state_dict(), f)
                print(f'  \u2713 checkpoint -> {ckpt} (model + optimizer + scheduler)')

                if LOG_CI_AT_CKPT:
                    # Informational only — never stops training (FIX 2).
                    m = evaluate_ci(model, processor, dev_samples,
                                    desc=f'CI @ step {global_step}')
                    print(f'  -> dev CI={m["contrastive_instability"]:.4f} '
                          f'(256px training config, informational)')

            if global_step >= MAX_STEPS:
                done = True
                break

    avg_epoch = epoch_loss / max(n_batches, 1)
    elapsed   = (time.time() - t0) / 3600
    print(f'\nEpoch {epoch} done | loss={avg_epoch:.4f} | elapsed={elapsed:.2f}h')

total_h = (time.time() - t0) / 3600
print(f'\nTraining complete at step {global_step}. Total: {total_h:.2f}h')
print('Now run Section 7 to pick the submission checkpoint.')

## 7. Checkpoint selection at deployment config (FIX 3)

Scores each candidate on dev at **1024px with the exact inference prompt + parser**
(what Codabench sees), not the 256px training config. Ties within `TIE_EPS` (2 dev
items) go to the **later** checkpoint — its LR is more annealed and dev CI differences
that small are noise. Winner is copied to `adapter_final` and zipped.

Runs standalone too: if the session died after training, upload
`/kaggle/working/checkpoints` as a dataset, set `RESUME_FROM` to the last checkpoint
with `RESUME_STEP = MAX_STEPS` (training cell then no-ops), and add the dataset path
to `CKPT_SEARCH_DIRS` below.

In [ ]:
from peft import load_peft_weights, set_peft_model_state_dict

# Where to look for candidate checkpoints (first hit per step wins).
CKPT_SEARCH_DIRS = [
    OUTPUT_DIR,
    # '/kaggle/input/<your-uploaded-checkpoints-dataset>/checkpoints',
]

candidates = {}
for step_n in CANDIDATE_STEPS:
    for d in CKPT_SEARCH_DIRS:
        p = os.path.join(d, f'step_{step_n}')
        if os.path.isdir(p):
            candidates[step_n] = p
            break
assert candidates, f'No candidate checkpoints found in {CKPT_SEARCH_DIRS}'
print(f'Candidates: { {k: v for k, v in sorted(candidates.items())} }')

# Deployment-resolution processor — matches the inference notebook exactly.
eval_processor = AutoProcessor.from_pretrained(VLM_MODEL, max_pixels=EVAL_MAX_PIXELS)

results = {}
for step_n in sorted(candidates):
    ckpt = candidates[step_n]
    print(f'\n=== step_{step_n} — loading adapter weights ===')
    set_peft_model_state_dict(model, load_peft_weights(ckpt))
    m = evaluate_ci(model, eval_processor, dev_samples,
                    inference_style=True, desc=f'step_{step_n} @1024px')
    results[step_n] = m
    print(f'step_{step_n}: dev CI={m["contrastive_instability"]:.4f} '
          f'(combined_acc={m["combined_accuracy"]:.4f}, '
          f'q+={m["q_plus_accuracy"]:.4f}, q-={m["q_minus_accuracy"]:.4f})')

# Pick: lowest CI; anything within TIE_EPS of the best is a tie -> latest step.
best_ci = min(m['contrastive_instability'] for m in results.values())
tied    = [s for s, m in results.items()
           if m['contrastive_instability'] <= best_ci + TIE_EPS]
winner  = max(tied)

print(f'\nBest dev CI @1024px: {best_ci:.4f}')
print(f'Within TIE_EPS ({TIE_EPS}): steps {sorted(tied)}')
print(f'WINNER: step_{winner} '
      f'(CI={results[winner]["contrastive_instability"]:.4f})')

# Copy winner to FINAL_ADAPTER (drop training state) and leave its weights
# loaded in `model` so any further cells use the submission adapter.
set_peft_model_state_dict(model, load_peft_weights(candidates[winner]))
os.makedirs(FINAL_ADAPTER, exist_ok=True)
for fname in os.listdir(candidates[winner]):
    if fname in ('optimizer.pt', 'scheduler.json'):
        continue
    shutil.copy2(os.path.join(candidates[winner], fname),
                 os.path.join(FINAL_ADAPTER, fname))
print(f'Copied step_{winner} -> {FINAL_ADAPTER}')

## 8. Zip final adapter

In [ ]:
import glob, zipfile

files    = sorted(glob.glob(f'{FINAL_ADAPTER}/*'))
assert files, 'FINAL_ADAPTER is empty — run Section 7 first.'
zip_path = '/kaggle/working/adapter_final.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in files:
        zf.write(f, os.path.basename(f))
        print(f'  {os.path.basename(f):<42} {os.path.getsize(f)/1024/1024:.1f} MB')

print(f'\nFinal adapter: {zip_path} '
      f'({os.path.getsize(zip_path)/1024/1024:.1f} MB)')
print('Download from Kaggle output panel.')
print('Use inference-qlora-q7b-frn.ipynb to run inference (MAX_PIXELS=1024*28*28).')

## 9. GPU memory summary

In [ ]:
for i in range(torch.cuda.device_count()):
    a = torch.cuda.max_memory_allocated(i)/1024**3
    r = torch.cuda.max_memory_reserved(i)/1024**3
    t = torch.cuda.get_device_properties(i).total_memory/1024**3
    print(f'GPU {i}: peak={a:.2f}GB  reserved={r:.2f}GB  total={t:.2f}GB')